<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Feature Detection and Object Tracking — Implementation</b></h1>
</div>

This notebook executes the complete fixed-reference feature-tracking experiment, including ORB extraction, Hamming-distance matching, RANSAC homography estimation, projected object localization, sequence-level diagnostics, figure generation, and final numerical validation.


## Setup — Environment and Configuration

Import the required libraries, configure compact numerical output, and create the shared figures directory used throughout the experiment.

In [1]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(
    precision=4,
    suppress=True,
)

OUTPUT_DIR = Path("../outputs/figures")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"OpenCV version: {cv2.__version__}")

OpenCV version: 5.0.0


## 1. Validate the Input Video and Output Paths

Verify that the repository-local input video exists and that the output directory is available before initializing the tracking pipeline.

In [2]:
VIDEO_PATH = Path("../data/video1.mp4")

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH}"
    )

if not VIDEO_PATH.is_file():
    raise ValueError(
        f"Input path is not a file: {VIDEO_PATH}"
    )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Input video: {VIDEO_PATH}")
print(f"Figures directory: {OUTPUT_DIR}")

Input video: ../data/video1.mp4
Figures directory: ../outputs/figures


## 2. Define the Initial Bounding Box and Tracking Parameters

Define the manually provided object region in the reference frame and the fixed ORB/RANSAC parameters used throughout the sequence.

In [3]:
ROW = 24
COL = 46
HEIGHT = 170
WIDTH = 160

MAX_FEATURES = 1000
RANSAC_THRESHOLD = 5.0
MIN_MATCHES = 4
MIN_INLIERS = 4

bbox = np.float32(
    [
        [COL, ROW],
        [COL + WIDTH, ROW],
        [COL + WIDTH, ROW + HEIGHT],
        [COL, ROW + HEIGHT],
    ]
).reshape(-1, 1, 2)

if bbox.shape != (4, 1, 2):
    raise ValueError(
        f"Unexpected bounding-box shape: {bbox.shape}"
    )

print("Initial bounding box corners:")
print(bbox.reshape(-1, 2))
print(
    f"ORB features: {MAX_FEATURES} | "
    f"RANSAC threshold: {RANSAC_THRESHOLD:.1f} px"
)

Initial bounding box corners:
[[ 46.  24.]
 [206.  24.]
 [206. 194.]
 [ 46. 194.]]
ORB features: 1000 | RANSAC threshold: 5.0 px


## 3. Initialize ORB and the Hamming-Distance Matcher

Create the ORB detector/descriptor and the brute-force binary-descriptor matcher used for all reference-to-frame correspondences.

In [4]:
orb = cv2.ORB_create(
    nfeatures=MAX_FEATURES,
)

# Hamming distance matches ORB's binary descriptors; cross-check removes one-way matches.
matcher = cv2.BFMatcher(
    cv2.NORM_HAMMING,
    crossCheck=True,
)

print("ORB detector initialized.")
print("BFMatcher metric: Hamming | crossCheck=True")

ORB detector initialized.
BFMatcher metric: Hamming | crossCheck=True


## 4. Read and Validate the Reference Frame

Read the first video frame, convert it to grayscale, and verify that the predefined bounding box lies completely inside the image.

In [5]:
cap = cv2.VideoCapture(
    str(VIDEO_PATH)
)

if not cap.isOpened():
    raise RuntimeError(
        f"Could not open video: {VIDEO_PATH}"
    )

reported_frame_count = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

ret, reference_frame = cap.read()
cap.release()

if not ret or reference_frame is None:
    raise RuntimeError(
        "Could not read the first video frame."
    )

reference_gray = cv2.cvtColor(
    reference_frame,
    cv2.COLOR_BGR2GRAY,
)

frame_height, frame_width = (
    reference_gray.shape
)

if (
    ROW < 0
    or COL < 0
    or ROW + HEIGHT > frame_height
    or COL + WIDTH > frame_width
):
    raise ValueError(
        "The initial bounding box lies outside "
        "the reference frame."
    )

print(
    f"Reference frame shape: "
    f"{reference_frame.shape}"
)
print(
    f"Reported video frames: "
    f"{reported_frame_count}"
)

Reference frame shape: (240, 320, 3)
Reported video frames: 851


### Interpretation

The video opens correctly, the first frame has shape **240×320×3**, and the reported sequence length is 851 frames. This confirms that the input can support the intended frame-by-frame tracking experiment and that the initialized bounding box lies in a valid image coordinate system.


## 5. Detect Reference ORB Features Inside the Object Region

Restrict feature extraction to the known object ROI in the first frame and validate that enough binary descriptors are available for homography estimation.

In [6]:
# Restrict reference features to the initialized object, not the full frame.
reference_mask = np.zeros(
    reference_gray.shape,
    dtype=np.uint8,
)

reference_mask[
    ROW:ROW + HEIGHT,
    COL:COL + WIDTH,
] = 255

reference_keypoints, reference_descriptors = (
    orb.detectAndCompute(
        reference_gray,
        reference_mask,
    )
)

if (
    reference_descriptors is None
    or len(reference_keypoints) < MIN_MATCHES
):
    raise RuntimeError(
        "Not enough ORB features were detected "
        "inside the initial bounding box."
    )

if (
    reference_descriptors.ndim != 2
    or reference_descriptors.shape[1] != 32
):
    raise ValueError(
        "Unexpected ORB descriptor shape: "
        f"{reference_descriptors.shape}"
    )

print(
    f"Reference keypoints detected: "
    f"{len(reference_keypoints)}"
)
print(
    f"Descriptor shape: "
    f"{reference_descriptors.shape}"
)

Reference keypoints detected: 868
Descriptor shape: (868, 32)


### Interpretation

The reference object region yields **868 ORB keypoints** with 32-byte binary descriptors. This is a dense feature set relative to the object area, giving the matcher substantial redundancy. Such redundancy is valuable because the tracker does not need every descriptor to survive viewpoint or appearance changes; it only needs enough geometrically consistent correspondences to estimate a homography.


## 6. Visualize the Reference Object and ORB Keypoints

Draw the initial object boundary together with the ORB keypoints retained inside the reference ROI and save the first diagnostic figure.

In [7]:
reference_with_bbox = (
    reference_frame.copy()
)

cv2.polylines(
    reference_with_bbox,
    [np.int32(bbox)],
    True,
    (0, 255, 0),
    3,
    cv2.LINE_AA,
)

reference_features = cv2.drawKeypoints(
    reference_with_bbox,
    reference_keypoints,
    None,
    flags=(
        cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    ),
)

reference_features_rgb = cv2.cvtColor(
    reference_features,
    cv2.COLOR_BGR2RGB,
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.imshow(
    reference_features_rgb
)
ax.set_title(
    "Reference Object and ORB Keypoints"
)
ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "reference_orb_keypoints.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 7. Define Frame Matching and RANSAC Homography Estimation

Implement the reference-to-current-frame matching function: detect ORB features, match binary descriptors with Hamming distance, estimate the projective mapping with RANSAC, and reject geometrically invalid solutions.

In [8]:
def match_and_estimate_homography(
    gray_frame,
    reference_keypoints,
    reference_descriptors,
    orb_detector,
    descriptor_matcher,
    min_matches=MIN_MATCHES,
    min_inliers=MIN_INLIERS,
    ransac_threshold=RANSAC_THRESHOLD,
):
    """Match ORB features and estimate a robust reference-to-frame homography."""
    keypoints, descriptors = (
        orb_detector.detectAndCompute(
            gray_frame,
            None,
        )
    )

    if (
        descriptors is None
        or len(keypoints) < min_matches
    ):
        return {
            "success": False,
            "reason": "insufficient_current_features",
            "num_matches": 0,
            "num_inliers": 0,
        }

    matches = descriptor_matcher.match(
        reference_descriptors,
        descriptors,
    )

    matches = sorted(
        matches,
        key=lambda match: match.distance,
    )

    if len(matches) < min_matches:
        return {
            "success": False,
            "reason": "insufficient_matches",
            "num_matches": len(matches),
            "num_inliers": 0,
        }

    matched_reference_points = np.float32(
        [
            reference_keypoints[
                match.queryIdx
            ].pt
            for match in matches
        ]
    ).reshape(-1, 2)

    matched_current_points = np.float32(
        [
            keypoints[
                match.trainIdx
            ].pt
            for match in matches
        ]
    ).reshape(-1, 2)

    # RANSAC separates geometrically consistent matches before accepting a homography.
    homography, inlier_mask = (
        cv2.findHomography(
            matched_reference_points,
            matched_current_points,
            cv2.RANSAC,
            ransac_threshold,
        )
    )

    if (
        homography is None
        or inlier_mask is None
        or homography.shape != (3, 3)
        or not np.all(
            np.isfinite(homography)
        )
    ):
        return {
            "success": False,
            "reason": "homography_failed",
            "num_matches": len(matches),
            "num_inliers": 0,
        }

    inlier_mask = (
        inlier_mask
        .ravel()
        .astype(bool)
    )

    num_inliers = int(
        inlier_mask.sum()
    )

    # Reject a projective update when geometric support is too weak.
    if num_inliers < min_inliers:
        return {
            "success": False,
            "reason": "insufficient_inliers",
            "num_matches": len(matches),
            "num_inliers": num_inliers,
        }

    return {
        "success": True,
        "reason": None,
        "keypoints": keypoints,
        "matches": matches,
        "homography": homography,
        "inlier_mask": inlier_mask,
        "num_matches": len(matches),
        "num_inliers": num_inliers,
    }


print(
    "Frame matching and homography "
    "estimation function defined."
)

Frame matching and homography estimation function defined.


## 8. Track the Object Throughout the Video

Use the first frame as a fixed reference, estimate one homography per subsequent frame, transform the original bounding box, and retain only compact geometric/statistical results in memory.

In [9]:
cap = cv2.VideoCapture(
    str(VIDEO_PATH)
)

if not cap.isOpened():
    raise RuntimeError(
        f"Could not open video: {VIDEO_PATH}"
    )

tracking_results = []
frame_index = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Skip the reference frame; tracking starts from the next observation.
    if frame_index == 0:
        frame_index += 1
        continue

    gray_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY,
    )

    result = match_and_estimate_homography(
        gray_frame=gray_frame,
        reference_keypoints=reference_keypoints,
        reference_descriptors=reference_descriptors,
        orb_detector=orb,
        descriptor_matcher=matcher,
    )

    # Store only diagnostics and geometry needed later to keep sequence memory bounded.
    compact_result = {
        "frame_index": frame_index,
        "success": result["success"],
        "reason": result["reason"],
        "num_matches": result["num_matches"],
        "num_inliers": result["num_inliers"],
    }

    if result["success"]:
        # Propagate the original object box with the reference-to-frame homography.
        current_bbox = (
            cv2.perspectiveTransform(
                bbox,
                result["homography"],
            )
        )

        if not np.all(
            np.isfinite(current_bbox)
        ):
            compact_result[
                "success"
            ] = False
            compact_result[
                "reason"
            ] = "invalid_projected_bbox"
        else:
            compact_result[
                "homography"
            ] = result["homography"]
            compact_result[
                "current_bbox"
            ] = current_bbox

    tracking_results.append(
        compact_result
    )

    frame_index += 1

cap.release()

successful_results = [
    result
    for result in tracking_results
    if result["success"]
]

failed_results = [
    result
    for result in tracking_results
    if not result["success"]
]

print(
    f"Frames processed: "
    f"{len(tracking_results)}"
)
print(
    f"Successful tracking frames: "
    f"{len(successful_results)}"
)
print(
    f"Failed tracking frames: "
    f"{len(failed_results)}"
)

Frames processed: 850
Successful tracking frames: 850
Failed tracking frames: 0


### Interpretation

Tracking succeeds on all **850 processed frames**, with no rejected frame under the current validity rules. This demonstrates that the fixed-reference ORB + Hamming + RANSAC pipeline remains operational throughout the full sequence rather than only over a short temporal window.


## 9. Compute Tracking Summary Metrics

Quantify the descriptor matching and geometric-consistency behavior of all successfully tracked frames using match count, inlier count, and RANSAC inlier ratio.

In [10]:
if not successful_results:
    raise RuntimeError(
        "No frame produced a valid homography. "
        "Check the video, ROI, or matching parameters."
    )

frame_indices = np.asarray(
    [
        result["frame_index"]
        for result in successful_results
    ],
    dtype=int,
)

match_counts = np.asarray(
    [
        result["num_matches"]
        for result in successful_results
    ],
    dtype=int,
)

inlier_counts = np.asarray(
    [
        result["num_inliers"]
        for result in successful_results
    ],
    dtype=int,
)

inlier_ratios = (
    inlier_counts
    / match_counts
)

success_rate = (
    len(successful_results)
    / len(tracking_results)
)

print(
    f"Tracking success rate: "
    f"{success_rate:.3f}"
)
print(
    f"Mean matches per successful frame: "
    f"{match_counts.mean():.2f}"
)
print(
    f"Mean RANSAC inliers per successful frame: "
    f"{inlier_counts.mean():.2f}"
)
print(
    f"Mean inlier ratio: "
    f"{inlier_ratios.mean():.3f}"
)
print(
    f"Minimum inlier ratio: "
    f"{inlier_ratios.min():.3f}"
)
print(
    f"Maximum inlier ratio: "
    f"{inlier_ratios.max():.3f}"
)

Tracking success rate: 1.000
Mean matches per successful frame: 381.80
Mean RANSAC inliers per successful frame: 290.49
Mean inlier ratio: 0.701
Minimum inlier ratio: 0.040
Maximum inlier ratio: 0.994


### Interpretation

The sequence averages about **381.8 descriptor matches** and **290.5 RANSAC inliers** per successful frame, with a mean inlier ratio of **0.701**. That means most matched features are usually consistent with one projective motion model. The minimum inlier ratio of **0.040**, however, shows that some frames are much more weakly supported than the average; successful tracking should therefore be judged from both the homography validity checks and the inlier statistics, not from the 100% success count alone.


## 10. Visualize Representative Tracking Frames

Reload only a small set of successful frames, draw the corresponding transformed bounding boxes, and save a representative tracking montage.

In [11]:
def read_video_frame(
    video_path,
    target_index,
):
    """Read one frame by index from the video."""
    capture = cv2.VideoCapture(
        str(video_path)
    )

    if not capture.isOpened():
        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    capture.set(
        cv2.CAP_PROP_POS_FRAMES,
        int(target_index),
    )

    ret, frame = capture.read()
    capture.release()

    if not ret or frame is None:
        raise RuntimeError(
            f"Could not read frame {target_index}."
        )

    return frame


n_examples = min(
    6,
    len(successful_results),
)

# Sample the sequence uniformly so diagnostics cover early, middle and late frames.
selection_indices = np.linspace(
    0,
    len(successful_results) - 1,
    n_examples,
    dtype=int,
)

selected_results = [
    successful_results[index]
    for index in selection_indices
]

n_cols = 3
n_rows = int(
    np.ceil(
        n_examples / n_cols
    )
)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, 5 * n_rows),
)

axes = np.asarray(
    axes
).reshape(-1)

for ax, result in zip(
    axes,
    selected_results,
):
    frame = read_video_frame(
        VIDEO_PATH,
        result["frame_index"],
    )

    cv2.polylines(
        frame,
        [
            np.int32(
                result["current_bbox"]
            )
        ],
        True,
        (0, 255, 0),
        3,
        cv2.LINE_AA,
    )

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB,
    )

    ax.imshow(
        frame_rgb
    )
    ax.set_title(
        f"Frame {result['frame_index']} | "
        f"Inliers: "
        f"{result['num_inliers']}/"
        f"{result['num_matches']}"
    )
    ax.axis("off")

for ax in axes[n_examples:]:
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "representative_tracking_frames.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 11. Visualize RANSAC Inlier Matches

Recompute the full correspondence set for a representative successful frame and display only the matches retained as geometric inliers by RANSAC.

In [12]:
example_result = (
    successful_results[
        len(successful_results) // 2
    ]
)

example_frame = read_video_frame(
    VIDEO_PATH,
    example_result["frame_index"],
)

example_gray = cv2.cvtColor(
    example_frame,
    cv2.COLOR_BGR2GRAY,
)

example_match_result = (
    match_and_estimate_homography(
        gray_frame=example_gray,
        reference_keypoints=reference_keypoints,
        reference_descriptors=reference_descriptors,
        orb_detector=orb,
        descriptor_matcher=matcher,
    )
)

if not example_match_result[
    "success"
]:
    raise RuntimeError(
        "Representative frame could not be "
        "re-matched for visualization."
    )

# Visualize only RANSAC inliers to expose geometric support, not raw descriptor matches.
matches_mask = (
    example_match_result[
        "inlier_mask"
    ]
    .astype(np.uint8)
    .tolist()
)

reference_display = (
    reference_frame.copy()
)

cv2.polylines(
    reference_display,
    [np.int32(bbox)],
    True,
    (0, 255, 0),
    3,
    cv2.LINE_AA,
)

current_display = (
    example_frame.copy()
)

cv2.polylines(
    current_display,
    [
        np.int32(
            example_result[
                "current_bbox"
            ]
        )
    ],
    True,
    (0, 255, 0),
    3,
    cv2.LINE_AA,
)

match_visualization = (
    cv2.drawMatches(
        reference_display,
        reference_keypoints,
        current_display,
        example_match_result[
            "keypoints"
        ],
        example_match_result[
            "matches"
        ],
        None,
        matchColor=(0, 255, 0),
        singlePointColor=None,
        matchesMask=matches_mask,
        flags=(
            cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        ),
    )
)

match_visualization_rgb = (
    cv2.cvtColor(
        match_visualization,
        cv2.COLOR_BGR2RGB,
    )
)

fig, ax = plt.subplots(
    figsize=(16, 7)
)

ax.imshow(
    match_visualization_rgb
)
ax.set_title(
    "RANSAC Inlier Matches — "
    f"Frame {example_result['frame_index']}"
)
ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "ransac_inlier_matches.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 12. Analyze Matches, Inliers, and Inlier Ratio Across the Sequence

Plot the temporal evolution of raw descriptor matches, RANSAC inliers, and their ratio to expose changes in correspondence quality throughout the video.

In [13]:
fig, ax = plt.subplots(
    figsize=(11, 5)
)

ax.plot(
    frame_indices,
    match_counts,
    marker=".",
    label="Descriptor matches",
)

ax.plot(
    frame_indices,
    inlier_counts,
    marker=".",
    label="RANSAC inliers",
)

ax.set_title(
    "Feature Matches and RANSAC Inliers by Frame"
)
ax.set_xlabel(
    "Frame index"
)
ax.set_ylabel(
    "Count"
)
ax.grid(
    alpha=0.25
)
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "matches_and_inliers_by_frame.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

fig, ax = plt.subplots(
    figsize=(11, 5)
)

ax.plot(
    frame_indices,
    inlier_ratios,
    marker=".",
)

ax.set_title(
    "RANSAC Inlier Ratio by Frame"
)
ax.set_xlabel(
    "Frame index"
)
ax.set_ylabel(
    "Inlier ratio"
)
ax.set_ylim(
    0,
    1.05,
)
ax.grid(
    alpha=0.25
)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "inlier_ratio_by_frame.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

### Interpretation

The temporal match/inlier curves are diagnostic rather than decorative: stable high values indicate strong appearance and geometric consistency, whereas sharp drops reveal difficult frames, viewpoint changes, blur, occlusion, or descriptor ambiguity. The inlier ratio is especially useful because it normalizes geometric support by the raw number of matches.


## 13. Run Numerical and Output-file Validation Checks

Validate the reference descriptors, processed-frame accounting, every accepted homography/bounding box, tracking statistics, and all five required diagnostic figures.

In [14]:
if (
    reference_descriptors is None
    or reference_descriptors.shape[0]
    < MIN_MATCHES
    or reference_descriptors.shape[1]
    != 32
):
    raise ValueError(
        "Invalid reference ORB descriptors."
    )

if len(tracking_results) == 0:
    raise ValueError(
        "No video frames were processed "
        "after the reference frame."
    )

if (
    reported_frame_count > 0
    and len(tracking_results)
    != reported_frame_count - 1
):
    raise ValueError(
        "Processed-frame count does not match "
        "the video frame count."
    )

for result in successful_results:
    H = result["homography"]
    current_bbox = result[
        "current_bbox"
    ]

    if (
        H.shape != (3, 3)
        or not np.all(
            np.isfinite(H)
        )
    ):
        raise ValueError(
            "Invalid homography in "
            f"frame {result['frame_index']}."
        )

    if (
        current_bbox.shape
        != (4, 1, 2)
        or not np.all(
            np.isfinite(current_bbox)
        )
    ):
        raise ValueError(
            "Invalid transformed bounding box "
            f"in frame {result['frame_index']}."
        )

    if (
        result["num_matches"]
        < MIN_MATCHES
        or result["num_inliers"]
        < MIN_INLIERS
        or result["num_inliers"]
        > result["num_matches"]
    ):
        raise ValueError(
            "Invalid correspondence statistics "
            f"in frame {result['frame_index']}."
        )

if not np.all(
    np.isfinite(
        inlier_ratios
    )
):
    raise ValueError(
        "Non-finite inlier ratio detected."
    )

if (
    np.any(inlier_ratios < 0.0)
    or np.any(inlier_ratios > 1.0)
):
    raise ValueError(
        "Inlier ratios outside [0, 1]."
    )

REQUIRED_OUTPUTS = [
    "reference_orb_keypoints.png",
    "representative_tracking_frames.png",
    "ransac_inlier_matches.png",
    "matches_and_inliers_by_frame.png",
    "inlier_ratio_by_frame.png",
]

missing_outputs = [
    name
    for name in REQUIRED_OUTPUTS
    if not (
        OUTPUT_DIR / name
    ).exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: "
        + ", ".join(
            missing_outputs
        )
    )

print(
    "All Feature Detection validation checks passed."
)

All Feature Detection validation checks passed.


### Interpretation

All Feature Tracking validation checks pass. The saved diagnostics, complete-frame processing, and finite correspondence statistics confirm that the implementation is internally consistent for this sequence under the fixed-reference homography model.


## Final Result Summary

The notebook implements a complete fixed-reference feature-based tracking pipeline using ORB keypoints/descriptors, Hamming-distance matching, RANSAC homography estimation, perspective transformation of the initial object box, sequence-level correspondence diagnostics, and explicit final validation.